In [0]:
# Databricks notebook source
# Camada Silver - Arquitetura Medalhão

from datetime import datetime

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# Definição de parâmetros
dbutils.widgets.text("catalogo", "ecommerce")
dbutils.widgets.text("schema_gold", "gold")
dbutils.widgets.text("schema_silver", "silver")

catalogo = dbutils.widgets.get("catalogo")
schema_silver = dbutils.widgets.get("schema_silver")
schema_gold = dbutils.widgets.get("schema_gold")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.{schema_gold}")

DataFrame[]

## 1º Projeto: Visão Comercial e Volume de Produtos

### Entrega 1 - Tabela Principal (gold.fat_vendas_comercial)

In [0]:
CATALOGO = "ecommerce"
SILVER = f"{CATALOGO}.silver"
GOLD   = f"{CATALOGO}.gold"

df_pedido_total = (
    spark.table(f"{SILVER}.fat_pedido_total")
    .select("id_pedido", F.to_date("data_pedido").alias("data_pedido"), "valor_total_pago_brl")
)

df_itens = (
    spark.table(f"{SILVER}.fat_itens_pedidos")
    .select("id_pedido", "id_item", "id_produto", "preco_BRL", "preco_frete")
)

df_prod = (
    spark.table(f"{SILVER}.dim_produtos")
    .select("id_produto", "categoria_produto")
)

df_cot = (
    spark.table(f"{SILVER}.dim_cotacao_dolar")
    .select(
        F.to_date("data_cotacao").alias("data_cotacao"),
        F.col("cotacao_dolar").cast("double").alias("cotacao_dolar")
    )
)

df_item = (
    df_itens.join(df_pedido_total, on="id_pedido", how="inner")
            .join(df_prod, on="id_produto", how="left")
            .join(df_cot, df_pedido_total["data_pedido"] == df_cot["data_cotacao"], how="left")
            .withColumn(
                "valor_item_brl",
                F.coalesce(F.col("preco_BRL").cast("double"), F.lit(0.0)) +
                F.coalesce(F.col("preco_frete").cast("double"), F.lit(0.0))
            )
            .withColumn(
                "valor_item_usd",
                F.when(
                    F.col("cotacao_dolar").isNotNull() & (F.col("cotacao_dolar") > 0),
                    F.col("valor_item_brl") / F.col("cotacao_dolar")
                )
            )
            .withColumn("ano_venda", F.year(df_pedido_total["data_pedido"]))
            .withColumn("mes_venda", F.month(df_pedido_total["data_pedido"]))
)

# ENTREGA 1 - agregação Ano/Mês/Categoria
df_fat_vendas_comercial = (
    df_item
    .groupBy("ano_venda", "mes_venda", "categoria_produto")
    .agg(
        F.countDistinct("id_pedido").alias("total_pedidos"),
        F.count("*").alias("qtd_itens_vendidos"),
        F.round(F.sum("valor_item_brl"), 2).alias("receita_total_brl"),
        F.round(F.sum("valor_item_usd"), 2).alias("receita_total_usd"),
    )
    .withColumn("ticket_medio_brl", F.round(F.col("receita_total_brl") / F.col("total_pedidos"), 2))
    .orderBy("ano_venda", "mes_venda", "categoria_produto")
)

# Gravar na Gold
(
    df_fat_vendas_comercial.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.fat_vendas_comercial")
)


In [0]:
display(df_fat_vendas_comercial)

ano_venda,mes_venda,categoria_produto,total_pedidos,qtd_itens_vendidos,receita_total_brl,receita_total_usd,ticket_medio_brl
2016,9,beleza_saude,1,3,143.46,43.06,143.46
2016,9,moveis_decoracao,1,2,136.23,42.01,136.23
2016,9,telefonia,1,1,75.06,22.94,75.06
2016,10,null,2,2,95.92,29.81,47.96
2016,10,alimentos,1,1,96.23,29.96,96.23
2016,10,audio,2,2,183.03,56.97,91.52
2016,10,automotivo,11,12,2257.56,699.94,205.23
2016,10,bebes,11,14,1819.08,564.09,165.37
2016,10,beleza_saude,44,48,5493.38,1707.11,124.85
2016,10,brinquedos,25,27,4986.08,1547.26,199.44


### Entrega 2 - Rankings Comerciais (Exibição via display()):

#### Top 5 Produtos MAIS Vendidos

In [0]:
from pyspark.sql import functions as F

SILVER = "ecommerce.silver"

df_itens = (
    spark.table(f"{SILVER}.fat_itens_pedidos").alias("it")
    .select("id_produto")
)

df_prod = (
    spark.table(f"{SILVER}.dim_produtos").alias("pr")
    .select("id_produto", "nome_produto", "categoria_produto")
)

df_rank_produtos = (
    df_itens.join(df_prod, on="id_produto", how="left")
            .groupBy("nome_produto", "categoria_produto")
            .agg(F.count("*").alias("quantidade_vendida"))
)

In [0]:
df_top5_mais = (
    df_rank_produtos
    .orderBy(F.desc("quantidade_vendida"), F.asc("nome_produto"))
    .limit(5)
)

display(df_top5_mais)

nome_produto,categoria_produto,quantidade_vendida
Secador de Cabelo,beleza_saude,1076
Toalha de Banho,cama_mesa_banho,919
Protetor Solar,beleza_saude,917
Travesseiro,cama_mesa_banho,839
Colar,relogios_presentes,804


#### Top 5 Produtos MENOS Vendidos

In [0]:
df_top5_menos = (
    df_rank_produtos
    .orderBy(F.asc("quantidade_vendida"), F.asc("nome_produto"))
    .limit(5)
)

display(df_top5_menos)

nome_produto,categoria_produto,quantidade_vendida
"""Smart TV 50"""" Básico""",eletronicos,1
"""Smart TV 50"""" Master""",eletronicos,1
Acessório Padrão,artigos_de_festas,1
Acessório Padrão,livros_importados,1
Acessório Padrão,fashion_roupa_infanto_juvenil,1


## 2º Projeto: Satisfação de Clientes e Qualidade de Parceiros

### Entrega 1 - Tabela Principal (gold.fat_avaliacoes_clientes)

In [0]:
# 1) Ler Silver
df_reviews = (
    spark.table(f"{SILVER}.fat_avaliacoes_pedidos") 
    .select(
        F.col("id_pedido"),
        F.col("nota_avaliacao").cast("int").alias("nota_avaliacao")  
    )
)

df_itens = (
    spark.table(f"{SILVER}.fat_itens_pedidos")
    .select("id_pedido", "id_produto", "id_vendedor")  
)

df_prod = (
    spark.table(f"{SILVER}.dim_produtos")
    .select("id_produto", "categoria_produto", "nome_produto")
)

df_vend = (
    spark.table(f"{SILVER}.dim_vendedores")
    .select(
        "id_vendedor",
        F.col("nome_vendedor"),
        F.col("estado")
    )
)

# 2) Base no grão "item avaliado": (pedido + item/produto + vendedor)
# Observação: um pedido pode ter múltiplos itens; a nota do pedido será repetida por item.
df_base = (
    df_reviews
    .join(df_itens, on="id_pedido", how="inner")
    .join(df_prod, on="id_produto", how="left")
    .join(df_vend, on="id_vendedor", how="left")
    .filter(F.col("nota_avaliacao").isNotNull())
)

# 3) Agregação na granularidade pedida
df_fat_avaliacoes_clientes = (
    df_base
    .groupBy("categoria_produto", "nome_vendedor", "estado")
    .agg(
        F.count("*").alias("total_avaliacoes"),
        F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media"),
        F.sum(F.when(F.col("nota_avaliacao") >= 4, 1).otherwise(0)).alias("total_avaliacoes_positivas"),
        F.sum(F.when(F.col("nota_avaliacao") <= 2, 1).otherwise(0)).alias("total_avaliacoes_negativas"),
    )
    .withColumn(
        "percentual_satisfacao",
        F.round((F.col("total_avaliacoes_positivas") / F.col("total_avaliacoes")) * 100, 2)
    )
    .orderBy(F.desc("avaliacao_media"), F.desc("percentual_satisfacao"))
)

# 4) Salvar na Gold
(
    df_fat_avaliacoes_clientes.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.fat_avaliacoes_clientes")
)

### Entrega 2 - Rankings Comerciais (Exibição via display())

In [0]:
df_prod_scores = (
    df_base.groupBy("id_produto", "nome_produto", "categoria_produto")
           .agg(
               F.count("*").alias("total_avaliacoes"),
               F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media")
           )
)


#### Produto MAIS bem avaliado


In [0]:
display(
    df_prod_scores
      .orderBy(F.desc("avaliacao_media"), F.desc("total_avaliacoes"))
      .limit(1)
)

id_produto,nome_produto,categoria_produto,total_avaliacoes,avaliacao_media
37eb69aca8718e843d897aa7b82f462d,Kit de Ferramentas Dourado,ferramentas_jardim,15,5.0


#### Produto MENOS bem avaliado


In [0]:
display(
    df_prod_scores
      .orderBy(F.asc("avaliacao_media"), F.desc("total_avaliacoes"))
      .limit(1)
)

#### Vendedor MAIS bem avaliado


In [0]:
df_vend_scores = (
    df_base.groupBy("id_vendedor", "nome_vendedor")
           .agg(
               F.count("*").alias("total_avaliacoes"),
               F.round(F.avg("nota_avaliacao"), 2).alias("avaliacao_media")
           )
)

In [0]:
display(
    df_vend_scores
      .orderBy(F.desc("avaliacao_media"), F.desc("total_avaliacoes"))
      .limit(1)
)

id_vendedor,nome_vendedor,total_avaliacoes,avaliacao_media
48efc9d94a9834137efd9ea76b065a38,Luiz Otávio Abreu,32,5.0


#### Vendedor MENOS bem avaliado


In [0]:
display(
    df_vend_scores
      .orderBy(F.asc("avaliacao_media"), F.desc("total_avaliacoes"))
      .limit(1)
)

id_vendedor,nome_vendedor,total_avaliacoes,avaliacao_media
8d92f3ea807b89465643c219455e7369,Sra. Fernanda Santos,8,1.0
